# Getting started with ANTsTorch

Create an ANTs image, preserve its physical geometry when converting to NumPy,
and construct a small PyTorch tensor. This example uses synthetic data only.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import matplotlib.pyplot as plt
import ants
import antstorch

device = "cpu"
antstorch.set_default_device(device)

## An image is more than an array
ANTs images carry spacing, origin and direction. NumPy arrays do not retain these
automatically. Here the first array axis is the physical x axis.

In [ ]:
x, y = np.meshgrid(np.linspace(-1, 1, 48), np.linspace(-1, 1, 40), indexing="ij")
array = np.exp(-4 * (x*x + y*y)).astype("float32")
image = ants.from_numpy(array, spacing=(1.2, 1.5), origin=(10., -5.))
reconstructed = ants.from_numpy(image.numpy(), spacing=image.spacing,
                                origin=image.origin, direction=image.direction)
assert reconstructed.shape == image.shape
np.testing.assert_allclose(reconstructed.numpy(), array)
np.testing.assert_allclose(reconstructed.direction, image.direction)
assert reconstructed.spacing == image.spacing
assert reconstructed.origin == image.origin
print(image)

## Convert spatial axes for tensor operations
A scalar 2-D tensor has layout `(batch, channels, height, width)`.
ANTs/ITK uses `(x, y)`; transpose to `(y, x)` and retain the geometry separately.

In [ ]:
tensor = torch.from_numpy(image.numpy().T.copy())[None, None].to(device)
assert tuple(tensor.shape) == (1, 1, 40, 48)
assert torch.isfinite(tensor).all()
plt.imshow(tensor[0, 0].cpu(), origin="lower", cmap="gray")
plt.title("Synthetic image (voxel coordinates)")
plt.xlabel("x voxel"); plt.ylabel("y voxel")
plt.show()

Continue with `unet_2d.ipynb`, `n4_bias_correction.ipynb`, or
`syn_registration.ipynb`. The pretrained application notebooks download data,
templates and weights on first use; inspect their resource notes first.